# Classification Models for Predicting Cluster Labels from Gene Presence-Absence

In [ ]:
# Libraries
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import LabelEncoder
from sklearn.multioutput import ClassifierChain
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import classification_report, confusion_matrix


In [ ]:
# Set file paths
repo_root = Path.cwd().parent
CLUSTER_FILE = repo_root / "analysis" / "reportree" / "ridom_partitions.tsv"
PRESENCE_ABSENCE_FILE = repo_root / "analysis" / "panaroo" / "merged" / "gene_presence_absence.Rtab"

In [ ]:
# Load labels and features
clusters = pd.read_csv(CLUSTER_FILE, sep="\t", index_col=0)
presence_absence = pd.read_csv(PRESENCE_ABSENCE_FILE, sep="\t", index_col=0)


In [ ]:
# For the presence-absence matrix - set samples to rows and features to columns
pa_transposed = presence_absence.transpose()
# For the clusters - we only want the columns for single-6x1.0, single-88x1.0, and single-501x1.0
target_clusters = clusters[["single-6x1.0", "single-88x1.0", "single-501x1.0"]]
# Merge the clusters and presence-absence data on the sample names
merged_data = pa_transposed.merge(target_clusters, left_index=True, right_index=True)

In [ ]:
# Count the number of clusters at the 501x1.0 level that have less than 10 samples
cluster_counts = merged_data["single-501x1.0"].value_counts()
print("Number of clusters with less than 10 samples at 501x1.0 level:")
print(cluster_counts[cluster_counts < 10].shape[0])

# Get the number of samples in clusters with less than 10 samples at the 501x1.0 level 
print("Number of samples in clusters with less than 10 samples at 501x1.0 level:")
print(cluster_counts[cluster_counts < 10].sum())

# Remove clusters with less than 10 samples at the 501x1.0 level
clusters_to_remove = cluster_counts[cluster_counts < 10].index
filtered_data = merged_data[~merged_data["single-501x1.0"].isin(clusters_to_remove)]



## Train/validation/test splits

The first important step is to split the model into sets for training, validation, and testing. Here, we will use a 60/20/20 train/validation/test split. The models are trained on the training set, evaluated iteratively with the validation set, and only the final model is tested on the test set. Because of the extreme class imbalances in our cluster labels, we should use stratified rather than random splits, which ensure that the same proportion of each class is present in the training, validation, and test sets. To achieve this, we also remove all samples that belong to clusters with less than 10 members at the 501-allele single-linkage threshold, to avoid having not enough samples to have enough in the training/val set for 5-fold CV with one in each fold. 

In [ ]:
# Separate features and labels
X = filtered_data.drop(columns=["single-6x1.0", "single-88x1.0", "single-501x1.0"])
y = filtered_data[["single-6x1.0", "single-88x1.0", "single-501x1.0"]]


In [ ]:
# Split training/validation from testing data
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size = 0.2, random_state=7, stratify=filtered_data[["single-501x1.0"]])


In [ ]:
# Use the LabelEncoder on the labels at each threshold
label_encoders = {}
for col in y_train_val.columns:
    le = LabelEncoder()
    y_train_val[col] = le.fit_transform(y_train_val[col])
    label_encoders[col] = le

In [ ]:
# Remove zero-variance features from the training/validation data
selector = VarianceThreshold(threshold=0)
X_train_val_selected = selector.fit_transform(X_train_val)

In [ ]:
# Before modeling - how many features do we have before and after removing zero-variance features?
print(f"Number of features before removing zero-variance features: {X_train_val.shape[1]}")
print(f"Number of features after removing zero-variance features: {X_train_val_selected.shape[1]}") 

In [ ]:
# Implement a Classifier Chain using Logistic Regression with cross-validation and L1 regularization
# Since the clusters are imbalanced, we can set the class_weight parameter to 'balanced' to help address this issue
# Also, since the clusters are hierarchical, we can set the order of the chain to reflect this hierarchy (i.e. single-501x1.0 -> single-88x1.0 -> single-6x1.0)

classifier_chain = ClassifierChain(
    LogisticRegressionCV(Cs=[1e-4, 1e-3, 1e-2, 1e-1, 1e0], l1_ratios=[1.0], solver='saga', class_weight='balanced'), 
    order=[2, 1, 0])
classifier_chain.fit(X_train_val_selected, y_train_val)

# Save the model to a file
import joblib
model_path = repo_root / "analysis" / "models"/ "classifier_chain_model.joblib"
joblib.dump(classifier_chain, model_path)


In [ ]:
# Evaluate the model performance on the test set after applying the same preprocessing steps (i.e. removing zero-variance features and converting labels to binary format)
X_test_selected = selector.transform(X_test)
for col in y_test.columns:
    le = label_encoders[col]
    y_test[col] = le.transform(y_test[col])  
test_score = classifier_chain.score(X_test_selected, y_test)
print(f"Test set accuracy: {test_score:.4f}")



In [ ]:
# Predict the labels for the test set
y_pred = classifier_chain.predict(X_test_selected)



In [ ]:
# Generate a classification report for each threshold
report_path = repo_root / "analysis" / "models" / "classifier_chain_classification_reports.txt"

with open(report_path, "w") as f:
    for i, col in enumerate(y_train_val.columns):
        le = label_encoders[col]
        report = classification_report(
            y_test.iloc[:, i],
            y_pred[:, i],
            target_names=le.classes_.astype(str)
        )
        f.write(f"Classification report for {col}:\n")
        f.write(report)
        f.write("\n\n")

In [ ]:
# Generate a confusion matrix for each threshold and save as CSV files

conf_matrix_path = repo_root / "analysis" / "models" / "confusion_matrices"
conf_matrix_path.mkdir(exist_ok=True)

for i, col in enumerate(y_train_val.columns):
    le = label_encoders[col]
    cm = confusion_matrix(y_test.iloc[:, i], y_pred[:, i])
    
    # Save as CSV with cluster labels as row/column headers
    cm_df = pd.DataFrame(
        cm,
        index=le.classes_.astype(str),
        columns=le.classes_.astype(str)
    )
    cm_df.to_csv(conf_matrix_path / f"confusion_matrix_{col}.csv")